# Impute Missing Data and Research Question
- Author: Bryan Bravo
- Created: 2026-04-07
## Research Question:
Using crude oil spot prices, the Liner Shipping Connectivity Index (LSCI), global political violence events, and a geopolitical risk index, can we accurately estimate a country's foreign exchange (FX) reserves and its imports of goods and services using a Random Forest Regressor?

### Selected Variables
#### Relative (Target) Variable:
- **fx_reserves**: Continuous Variable
- **imports_good_service**: Continuous Variable

#### Explanatory Variables:
- **country**: Categorical Variable
- **date**: Categorical Variable
- **interest_rate**: Continuous Variable
- **fx_rate**: Continuous Variable
- **brent_dollars_per_barrel**: Continuous Variable
- **wti_dollars_per_barrel**: Continuous Variable
- **events**: Continuous Variable
- **cpi**: Continuous Variable
- **gpr_index**: Continuous Variable
- **lsci**: Continuous Variable

## Import Libraries

In [1]:
import os
import sys

os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")
# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# Visualize data for joined dataframes.
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn libraries and packages
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Variables

In [3]:
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
# in_path = 's3a://ml-project-s3-bronze/input_folder/'
in_path = 'processed_datasets/'
out_path = 'processed_datasets/'


## User Defined Functions 

# Query

In [4]:
# Import dataset
joined_df = spark.read.csv(in_path + 'joined_input.csv', header=True, inferSchema=True)
joined_df = joined_df.withColumn('date', F.to_date(F.col('date'), 'yyyyMMdd'))

In [5]:
# # Create separate dataframes for each country
# country_list = joined_df.select(F.col('country')).distinct().toPandas()['country'].tolist()
# country_dct = {f"{country}_df": joined_df.filter(F.col('country').contains(country)) 
#                for country in country_list}
# [*country_dct.keys()]

In [6]:
joined_df.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60276 entries, 0 to 60275
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   year                      60276 non-null  int32  
 1   month                     60276 non-null  int32  
 2   country                   60276 non-null  object 
 3   date                      60276 non-null  object 
 4   interest_rate             60276 non-null  float64
 5   fx_rate                   60276 non-null  float64
 6   brent_dollars_per_barrel  60276 non-null  float64
 7   wti_dollars_per_barrel    60276 non-null  float64
 8   events                    60276 non-null  int32  
 9   cpi                       60276 non-null  float64
 10  gpr_index                 60276 non-null  float64
 11  lsci                      60276 non-null  float64
 12  fx_reserves               55596 non-null  float64
 13  imports_good_service      55596 non-null  float64
dtypes: flo

## Data Wrangling
### One-hot encode `country`, `month` variables, and add a one-hot encoded variable for day-of-week.


In [7]:
# Extract country values
country_list = joined_df.select(F.col('country')).distinct().toPandas()['country'].tolist()

wrangled_df = (
    joined_df
    .withColumns({  # One-hot encode country column
        f"{country}_n": F.when(F.col('country').contains(country), F.lit(1)).otherwise(F.lit(0)) for country in country_list
    })
    .withColumns({  # One-hot encode month column
        f"month_{i}": F.when(F.col('month') == i, F.lit(1)).otherwise(F.lit(0)) for i in range(1, 13)
    })
    .withColumns({  # Generate a day of week one-hot encoding (1-7, where 1 = Sunday and 7 = Saturday)
        f'dow_{i}': F.when(F.dayofweek(F.col('date')) == i, F.lit(1)).otherwise(F.lit(0)) for i in range(1, 8)
    })
    .drop('country', 'month', 'date')
)

wrangled_null_response_df = (  # Isolating rows where response variables are null. Expected 4680 rows.
    wrangled_df.filter(F.col('fx_reserves').isNull() & F.col('imports_good_service').isNull())
).toPandas()

wrangled_df = (  # Will split train test with this dataset.
    wrangled_df.filter(F.col('fx_reserves').isNotNull() & F.col('imports_good_service').isNotNull())
).toPandas()
print("wrangled_df:", wrangled_df.shape)
print("wrangled_null_response_df:", wrangled_null_response_df.shape)

wrangled_df: (55596, 46)
wrangled_null_response_df: (4680, 46)


In [8]:
wrangled_df.head()

,year,interest_rate,fx_rate,brent_dollars_per_barrel,wti_dollars_per_barrel,events,cpi,gpr_index,lsci,fx_reserves,...,month_10,month_11,month_12,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6,dow_7
0,2006,5.728261,0.7519,67.57,71.42,0,84.5,0.062846,141.65,49615.006093,...,0,0,0,0,0,0,1,0,0,0
1,2006,5.728261,0.7619,69.82,71.85,0,84.5,0.062846,141.65,49615.006093,...,0,0,0,0,0,1,0,0,0,0
2,2006,5.728261,0.7577,69.88,71.35,0,84.5,0.062846,141.65,49615.006093,...,0,0,0,0,0,0,0,0,1,0
3,2006,5.728261,0.7585,68.51,70.92,0,84.5,0.062846,141.65,49615.006093,...,0,0,0,0,0,0,0,1,0,0
4,2006,5.728261,0.7526,68.45,69.47,0,84.5,0.062846,141.65,49615.006093,...,0,0,0,0,0,0,1,0,0,0


### Feature Selection using SelectKBest
`fx_reserves` response variable

In [32]:
# Define variables
response_var = 'fx_reserves'
explanatory_var = [var for var in wrangled_df.columns if var not in ['fx_reserves', 'imports_good_service']]

# Assign values
X = wrangled_df[explanatory_var]
y = wrangled_df[response_var]
X.shape, y.shape

((55596, 44), (55596,))

In [33]:
# Use SelectKBest and use the fit_transform method.
skbest = SelectKBest(k='all', score_func=f_regression)
X_skbest = skbest.fit_transform(X, y)
X_skbest.shape

(55596, 44)

In [34]:
# Finding p-values and dataframe creation.
pval_df = pd.DataFrame({
    'Feature': X.columns,
    'p_value': skbest.pvalues_
}).sort_values('p_value')

# Selected Features
pval_pf = pval_df[pval_df["p_value"] < 0.05]
fx_reserves_selected_features = pval_pf['Feature'].tolist()  # extract selected feature list.
print("fx_reserves selected feature count: ", len(fx_reserves_selected_features))
pval_pf.style.format({'Feature': '{}', 'p_value': '{:.5f}'})

fx_reserves selected feature count:  24


,Feature,p_value
8,lsci,0.00000
12,china_n,0.00000
11,japan_n,0.00000
18,south_africa_n,0.00000
13,australia_n,0.00000
16,france_n,0.00000
10,canada_n,0.00000
22,united_kingdom_n,0.00000
2,fx_rate,0.00000
24,united_states_n,0.00000


`imports_good_service` response variable

In [35]:
# Define variables
response_var = 'imports_good_service'
explanatory_var = [var for var in wrangled_df.columns if var not in ['fx_reserves', 'imports_good_service']]

# Assign values
X = wrangled_df[explanatory_var]
y = wrangled_df[response_var]
X.shape, y.shape

((55596, 44), (55596,))

In [36]:
# Use SelectKBest and use the fit_transform method.
skbest = SelectKBest(k='all', score_func=f_regression)
X_skbest = skbest.fit_transform(X, y)
X_skbest.shape

(55596, 44)

In [37]:
# Finding p-values and dataframe creation.
pval_df = pd.DataFrame({
    'Feature': X.columns,
    'p_value': skbest.pvalues_
}).sort_values('p_value')

# Selected Features
pval_pf = pval_df[pval_df["p_value"] < 0.05]
imports_good_service_selected_features = pval_pf['Feature'].tolist()  # extract selected feature list.
print("imports_good_service selected feature count: ", len(imports_good_service_selected_features))
pval_pf.style.format({'Feature': '{}', 'p_value': '{:.5f}'})

imports_good_service selected feature count:  25


,Feature,p_value
0,year,0.00000
1,interest_rate,0.00000
2,fx_rate,0.00000
24,united_states_n,0.00000
6,cpi,0.00000
7,gpr_index,0.00000
8,lsci,0.00000
9,brazil_n,0.00000
12,china_n,0.00000
13,australia_n,0.00000


# Machine Learning
## `fx_reserves`